In [1]:
import numpy as np
import librosa

def extract_stft(
    signal,
    sr=16000,
    n_fft=512,
    hop_length=160,   # 10 ms
    win_length=400    # 25 ms
):

    stft = librosa.stft(
        signal,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window="hann"
    )

    # magnitude spectrogram
    spec = np.abs(stft)

    # log scale
    log_spec = librosa.amplitude_to_db(spec)

    return log_spec.T   # shape: (frames, freq_bins)

In [2]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern


def build_stft_dataset(
    wav_list,
    sr=16000,
    segment_sec=2,
    out_path="X_stft.npy"
):

    all_feats = []
    segment_len = int(segment_sec * sr)

    for wav_path in wav_list:

        print("Processing:", wav_path)

        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:

            stft_feat = extract_stft(segment, sr)

            lbp = local_binary_pattern(
                stft_feat,
                P=8,
                R=1,
                method="default"
            )

            hist, _ = np.histogram(
                lbp.ravel(),
                bins=256,
                range=(0, 256)
            )

            all_feats.append(hist)

        print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [ ]:
import os
CLASS = "noqueen"
DIR = r"E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech"

wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".wav")
]
print(wav_list)
X_mfcc = build_stft_dataset(
    wav_list,
    out_path=f"E:/Pythonfile/Voice-Activity-Detect/data/feature_bee/stft_lbp_speech_aurora.npy"
)

print(len(wav_list))
print(X_mfcc.shape)

['E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\0.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\1.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\10.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\100.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\101.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\102.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\103.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\104.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\105.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\106.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\aurora\\speech-noise\\107.wav', 'E:\\Pythonfile\\Voice-Ac

e:\Pythonfile\Voice-Activity-Detect\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\0.wav -> 1 segments
Processing: E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\1.wav
E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\1.wav -> 1 segments
Processing: E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\10.wav
E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\10.wav -> 1 segments
Processing: E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\100.wav
E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\100.wav -> 1 segments
Processing: E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\101.wav
E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\101.wav -> 1 segments
Processing: E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\102.wav
E:\Pythonfile\Voice-Activity-Detect\data\processed\aurora\speech-noise\102.wav -> 1 segm